# ⚙️ Lucile Sport — Full Hybrid RAG Pipeline Verification
This notebook demonstrates and validates the complete **Hybrid RAG (Dense FAISS + Sparse BM25)** pipeline for **Lucile Sport (لوسيل سبورت)**.

### Features Tested:
1. **Dense Semantic Search**: FAISS index with `all-MiniLM-L6-v2` embeddings.
2. **Sparse Lexical Search**: BM25 keyword matching.
3. **Reciprocal Rank Fusion (RRF)**: Merging dense and sparse ranks.
4. **Egyptian Car Dialect Expansion**: Resolving queries like `اسبورتاج` ⟷ `Sportage`, `النترا` ⟷ `Elantra`.
5. **Installment Calculation Engine**: Dynamic 3, 6, 9, 12 month installment plans in EGP (ج.م).
6. **LLM Response Generation & Maintenance Handoff**.

In [1]:
import os
import sys
sys.path.append("..")

from core.data_loader import load_products_from_csv
from core.retrieval import SalesRetrievalEngine
from core.pipeline import RAGPipeline

# Load products
csv_path = "../data/products_clean.csv" if os.path.exists("../data/products_clean.csv") else "data/products_clean.csv"
products = load_products_from_csv(csv_path)
print(f"Loaded {len(products):,} active automotive products.")

## 1. Initialize Hybrid Search Engine (FAISS + BM25)

In [2]:
engine = SalesRetrievalEngine()
engine.load_indexes(products)
print("Search engine successfully initialized.")

## 2. Test Hybrid Search with Egyptian Dialect Queries

In [3]:
test_queries = [
    "عايز تيل فرامل كيا سبورتاج",
    "طنابير هيونداي فيرنا أمامي",
    "فلتر زيت تويوتا كورولا",
    "مساعدين نيسان صني"
]

for q in test_queries:
    results = engine.hybrid_search(q, top_k=3)
    print(f"\n🔍 Query: '{q}' -> Found {len(results)} items:")
    for idx, (p, score) in enumerate(results, 1):
        print(f"   {idx}. {p.get('title')[:60]} | {float(p.get('final_price', 0)):,.0f} ج.م | Score: {score:.4f}")

## 3. Test Egyptian Installment Policy Engine (EGP)

In [4]:
from core.pipeline import calculate_installment

sample_price = 5000.0  # 5,000 EGP
print(f"Installment Calculations for {sample_price:,.0f} ج.م:")
for months in [3, 6, 9, 12]:
    plan = calculate_installment(sample_price, months)
    print(f"  • {months} شهور: {plan['monthly_payment']:,.1f} ج.م/شهر (إجمالي: {plan['total_with_interest']:,.0f} ج.م بفائدة {int(plan['interest_rate']*100)}%)")

## 4. End-to-End RAG Pipeline Test

In [5]:
pipeline = RAGPipeline(engine)
response = pipeline.process_message("عايز طقم تيل فرامل ومساعدين لسيارة رينو ميجان", session_id="nb_test")
print("\n🤖 Assistant Response:\n")
print(response['message'])
print(f"\nAttached Product Cards: {len(response.get('products', []))}")